# ЛАНИТ / SOLUT: классификация рабочих операций по IMU

Демо-notebook по мотивам проекта из презентации: классификация действий сотрудника по данным носимого устройства.

Что внутри:
- синтетические акселерометр/гироскоп сигналы 50 Гц;
- windowing по коротким окнам;
- feature extraction для временных рядов;
- `RandomForestClassifier` для нижнего уровня действий;
- агрегация коротких действий в операции верхнего уровня.

Данные синтетические: они имитируют разные паттерны движения, но не являются реальными сенсорными данными.

## Demo scope и production scale

Этот notebook — демонстрационный baseline, который имитирует задачу классификации рабочих операций на маленьком синтетическом наборе. Здесь сигналы сгенерированы программно, классы хорошо разделимы, а модель обучается быстро в памяти ноутбука.

В реальном проекте система была бы заметно сложнее:
- реальные IMU-потоки 50 Гц от множества сотрудников, смен, объектов и типов устройств;
- синхронизация акселерометра, гироскопа, GPS, барометра, пульса и видеоразметки;
- ошибки разметки, шум датчиков, разное положение устройства на теле и drift между объектами;
- двухуровневая архитектура: короткие движения → операции 30-60 секунд → хронология рабочего дня;
- anomaly detection для опасных действий, нарушений ТБ и нетипичного поведения;
- мониторинг качества, latency, стабильности классов и переобучение под новые площадки.

Смысл demo: показать принцип windowing, feature extraction, классификации и агрегации операций без доступа к реальным сенсорным данным.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(42)

FS = 50
WINDOW_SECONDS = 5
N_STEPS = FS * WINDOW_SECONDS
LOW_LEVEL_CLASSES = ["rest", "walk", "hammer", "wrench", "carry"]


def simulate_window(label: str) -> np.ndarray:
    t = np.arange(N_STEPS) / FS
    noise = rng.normal(0, 0.08, size=(N_STEPS, 6))
    x = noise.copy()

    if label == "rest":
        x[:, 2] += 1.0
    elif label == "walk":
        x[:, 0] += 0.6 * np.sin(2 * np.pi * 1.8 * t)
        x[:, 2] += 1.0 + 0.35 * np.sin(2 * np.pi * 3.6 * t)
        x[:, 4] += 0.25 * np.sin(2 * np.pi * 1.8 * t)
    elif label == "hammer":
        pulses = (np.sin(2 * np.pi * 2.8 * t) > 0.82).astype(float)
        x[:, 1] += 2.2 * pulses
        x[:, 5] += 1.4 * pulses
        x[:, 2] += 1.0
    elif label == "wrench":
        x[:, 3] += 0.9 * np.sin(2 * np.pi * 1.2 * t)
        x[:, 4] += 0.9 * np.cos(2 * np.pi * 1.2 * t)
        x[:, 2] += 1.0
    elif label == "carry":
        x[:, 0] += 0.25 * np.sin(2 * np.pi * 1.4 * t)
        x[:, 2] += 1.3 + 0.15 * np.sin(2 * np.pi * 2.8 * t)
        x[:, 5] += 0.15
    return x


def make_dataset(n_per_class: int = 220):
    rows = []
    raw = []
    labels = []
    for label in LOW_LEVEL_CLASSES:
        for _ in range(n_per_class):
            arr = simulate_window(label)
            raw.append(arr)
            labels.append(label)
    return np.array(raw), np.array(labels)

raw_windows, y = make_dataset()
raw_windows.shape, y[:5]

## Аналитический вывод

Синтетические окна имитируют разные режимы движения: покой, ходьбу, удары молотком, вращение ключом и перенос груза. Частота 50 Гц и окно 5 секунд соответствуют постановке из презентации: короткий фрагмент содержит достаточно сигнала для распознавания элементарного действия.

В реальном проекте на этом этапе критичны синхронизация датчиков, очистка выбросов и качество видеоразметки, потому что ошибки в разметке напрямую ограничивают качество классификатора.

In [ ]:
def extract_features(windows: np.ndarray) -> pd.DataFrame:
    names = ["acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z"]
    rows = []
    for window in windows:
        features = {}
        magnitude = np.sqrt((window[:, :3] ** 2).sum(axis=1))
        gyro_magnitude = np.sqrt((window[:, 3:] ** 2).sum(axis=1))
        for idx, name in enumerate(names):
            values = window[:, idx]
            features[f"{name}_mean"] = values.mean()
            features[f"{name}_std"] = values.std()
            features[f"{name}_min"] = values.min()
            features[f"{name}_max"] = values.max()
            features[f"{name}_energy"] = np.mean(values ** 2)
        features["acc_mag_mean"] = magnitude.mean()
        features["acc_mag_std"] = magnitude.std()
        features["gyro_mag_mean"] = gyro_magnitude.mean()
        features["gyro_mag_std"] = gyro_magnitude.std()
        rows.append(features)
    return pd.DataFrame(rows)

X = extract_features(raw_windows)
X.head()

## Аналитический вывод

Из сырых временных рядов извлекаются агрегированные признаки: средние, стандартные отклонения, экстремумы, энергия и магнитуды ускорения/гироскопа. Это простой и интерпретируемый baseline для классификации сенсорных данных.

Такой подход быстрее и дешевле глубоких моделей, поэтому хорошо подходит как первый production baseline. Если данных станет больше, его можно заменить или дополнить CNN/TCN/Transformer-моделью по сырым окнам.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

clf = RandomForestClassifier(
    n_estimators=250,
    max_depth=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
clf.fit(X_train, y_train)

pred = clf.predict(X_test)
print(classification_report(y_test, pred))
print(pd.DataFrame(confusion_matrix(y_test, pred), index=LOW_LEVEL_CLASSES, columns=LOW_LEVEL_CLASSES))

## Аналитический вывод

Высокое качество на синтетическом test set означает, что выбранные признаки хорошо разделяют заданные паттерны движений. В реальных IMU-данных качество будет ниже из-за шума, различий между людьми, положения устройства, ошибок разметки и смены условий работы.

Для production-версии важно смотреть не только accuracy, но и `recall` по критичным классам: например, опасные действия или нарушения техники безопасности лучше пропускать как можно реже.

In [ ]:
UPPER_LEVEL_MAP = {
    "rest": "rest",
    "walk": "movement",
    "hammer": "productive_work",
    "wrench": "productive_work",
    "carry": "preparation",
}

# Имитация 60-секундного фрагмента рабочего дня: 12 окон по 5 секунд.
scenario = ["walk", "walk", "carry", "carry", "hammer", "hammer", "hammer", "wrench", "wrench", "rest", "rest", "walk"]
scenario_raw = np.array([simulate_window(label) for label in scenario])
scenario_X = extract_features(scenario_raw)
low_predictions = clf.predict(scenario_X)

operation_timeline = pd.DataFrame({
    "window_start_sec": np.arange(len(scenario)) * WINDOW_SECONDS,
    "true_low_level": scenario,
    "pred_low_level": low_predictions,
    "pred_upper_operation": [UPPER_LEVEL_MAP[x] for x in low_predictions],
})
operation_timeline

## Аналитический вывод

Эта ячейка показывает двухуровневую архитектуру из презентации: короткие окна сначала классифицируются в элементарные действия (`walk`, `hammer`, `wrench`, `carry`, `rest`), а затем маппятся в операции верхнего уровня (`movement`, `productive_work`, `preparation`, `rest`).

Такой подход удобен для промышленной аналитики: нижний уровень сохраняет детализацию движений, а верхний уровень дает понятную бизнес-интерпретацию рабочего дня.

In [ ]:
summary = (
    operation_timeline.groupby("pred_upper_operation")
    .size()
    .rename("n_windows")
    .reset_index()
)
summary["minutes"] = summary["n_windows"] * WINDOW_SECONDS / 60
summary["share"] = summary["n_windows"] / summary["n_windows"].sum()
summary.sort_values("minutes", ascending=False)

## Аналитический вывод

Итоговая сводка переводит поток сенсорных предсказаний в бизнес-язык: сколько времени сотрудник провел в продуктивной работе, перемещении, подготовке или отдыхе. Именно такой слой нужен заказчику, потому что отдельные движения сами по себе менее полезны, чем хронология рабочего дня и доли операций.

В production это можно использовать для поиска простоев, нарушений техники безопасности и зон, где процесс можно оптимизировать.